In [1]:
import numpy as np

In [2]:
np.random.seed(42)
conv_weights = np.random.randn(8, 3, 3, 3).astype(np.float32)
conv_weights[0] *= 0.1      
conv_weights[1] *= 0.5      
conv_weights[6] *= 5.0      
conv_weights[7] *= 10.0     

print("Tensor Shape:", conv_weights.shape)

Tensor Shape: (8, 3, 3, 3)


In [3]:
def per_tensor_quantize(tensor):
    """
    Performs symmetric per-tensor quantization.
    """
    max_abs = np.max(np.abs(tensor))
    scale = max_abs / 127
    quantized = np.round(tensor / scale)
    quantized = np.clip(quantized, -127, 127)
    quantized = quantized.astype(np.int8)
    dequantized = quantized.astype(np.float32) * scale
    return quantized, dequantized, scale

In [4]:
def per_channel_quantize(tensor):
    """
    Performs symmetric per-output-channel quantization.
    """
    quantized = np.zeros_like(tensor, dtype=np.int8)
    dequantized = np.zeros_like(tensor, dtype=np.float32)
    scales = []
    for c in range(tensor.shape[0]):
        channel = tensor[c]
        max_abs = np.max(np.abs(channel))
        scale = max_abs / 127
        scales.append(scale)
        q = np.round(channel / scale)
        q = np.clip(q, -127, 127)
        quantized[c] = q.astype(np.int8)
        dequantized[c] = quantized[c].astype(np.float32) * scale
    return quantized, dequantized, np.array(scales)

In [5]:
def calculate_mae(original, reconstructed):

    return np.mean(np.abs(original - reconstructed))

In [6]:
# Per-Tensor
pt_quantized, pt_dequantized, pt_scale = per_tensor_quantize(conv_weights)

# Per-Channel
pc_quantized, pc_dequantized, pc_scales = per_channel_quantize(conv_weights)

In [8]:
print("="*110)

print(f"{'Channel':<8}{'Range (Min, Max)':<30}{'PT Scale':<15}{'PT MAE':<15}{'PC Scale':<15}{'PC MAE':<15}{'Better'}")

print("="*110)

pt_mae_list = []

pc_mae_list = []

for c in range(conv_weights.shape[0]):

    channel = conv_weights[c]

    ch_min = np.min(channel)

    ch_max = np.max(channel)

    pt_mae = calculate_mae(channel, pt_dequantized[c])

    pc_mae = calculate_mae(channel, pc_dequantized[c])

    pt_mae_list.append(pt_mae)

    pc_mae_list.append(pc_mae)

    better = "Per-Channel" if pc_mae < pt_mae else "Per-Tensor"

    print(f"{c:<8}{f'({ch_min:.2f}, {ch_max:.2f})':<30}"
          f"{pt_scale:<15.6f}"
          f"{pt_mae:<15.6f}"
          f"{pc_scales[c]:<15.6f}"
          f"{pc_mae:<15.6f}"
          f"{better}")

Channel Range (Min, Max)              PT Scale       PT MAE         PC Scale       PC MAE         Better
0       (-0.19, 0.16)                 0.303365       0.071464       0.001507       0.000326       Per-Channel
1       (-0.98, 0.93)                 0.303365       0.072564       0.007715       0.001661       Per-Channel
2       (-2.62, 1.56)                 0.303365       0.072175       0.020628       0.005373       Per-Channel
3       (-1.46, 1.89)                 0.303365       0.071594       0.014852       0.003731       Per-Channel
4       (-1.92, 2.46)                 0.303365       0.070571       0.019396       0.003995       Per-Channel
5       (-1.61, 1.87)                 0.303365       0.068939       0.014691       0.003901       Per-Channel
6       (-5.35, 13.60)                0.303365       0.077800       0.107093       0.027714       Per-Channel
7       (-15.15, 38.53)               0.303365       0.064038       0.303365       0.064038       Per-Tensor


In [9]:
print("\n")

print("Average Per-Tensor MAE :", np.mean(pt_mae_list))

print("Average Per-Channel MAE:", np.mean(pc_mae_list))



Average Per-Tensor MAE : 0.07114315
Average Per-Channel MAE: 0.013842214


Observation

Per-channel quantization produced lower reconstruction errors for channels with smaller value ranges because each channel used its own scale. Per-tensor quantization used a single scale for the entire tensor, which reduced accuracy for channels with small values when larger-value channels were present.